In [2]:
!pip install anndata==0.8.0

In [40]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
import scipy

In [41]:
fn = '../../Active_SAM_joined/SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad'

In [42]:
sam = SAM()
sam.load_data(fn)

In [43]:
lsaturn_mv = pd.read_csv('../../SATURN_mapping/XT_vole_mouse_SATURN_30_seed.csv', index_col = 'Unnamed: 0')
lsaturn_m = pd.read_csv('../../SATURN_mapping/XT_30_seeds.csv', index_col = 'barcode')

In [44]:
ind = pd.read_csv('../../Active_SAMap_Joined/XT_MG_mapping_cleaned_03122025_0.csv')['Unnamed: 0']
lsamap_m = pd.DataFrame(index = list(ind))
for i in range(30):
    df = pd.read_csv('../../Active_SAMap_Joined/XT_MG_mapping_cleaned_03122025_'+str(i)+'.csv', index_col = 'Unnamed: 0')
    lsamap_m = pd.concat([lsamap_m, df], axis = 1)

In [45]:
lsamap_m

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
AAACCCAAGATCCAAA Run19_sample1,088 BST Tac2 Gaba,088 BST Tac2 Gaba,107 DMH Hmx2 Gaba,088 BST Tac2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba,...,088 BST Tac2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba,088 BST Tac2 Gaba
AAACCCAGTACGGATG Run19_sample1,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,...,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut,110 BST-po Iigp1 Glut
AAACCCAGTTACGTAC Run19_sample1,337 DC NN,337 DC NN,337 DC NN,337 DC NN,337 DC NN,337 DC NN,337 DC NN,337 DC NN,336 Monocytes NN,337 DC NN,...,337 DC NN,336 Monocytes NN,337 DC NN,337 DC NN,337 DC NN,337 DC NN,337 DC NN,337 DC NN,337 DC NN,336 Monocytes NN
AAACCCAGTTCAAACC Run19_sample1,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,...,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut,141 PH-SUM Foxa1 Glut
AAACCCATCAGGGTAG Run19_sample1,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,...,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTCATAGTC Run19_sample8,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,...,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba,094 SCH Six6 Cdc14a Gaba
TTTGTTGGTGAGAGGG Run19_sample8,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,037 DG Glut,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,037 DG Glut,037 DG Glut,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,...,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,037 DG Glut,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,015 ENTmv-PA-COAp Glut,037 DG Glut
TTTGTTGGTGTTACTG Run19_sample8,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,...,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba,214 IPN Otp Crisp1 Gaba
TTTGTTGTCGGCCCAA Run19_sample8,124 MPN-MPO-PVpo Hmx2 Glut,124 MPN-MPO-PVpo Hmx2 Glut,124 MPN-MPO-PVpo Hmx2 Glut,124 MPN-M

In [46]:
meta_folder = '../../Active_SAMap_Joined/XT_metadata/'

In [47]:
mn = os.listdir('../../Active_SAMap_Joined/XT_metadata/')

In [48]:
pd.read_csv(meta_folder + mn[3])

,Unnamed: 0,n_genes,n_counts,key,leiden_clusters,subclass_id_label_mapping,subclass_id_label_lc
0,AAACCCAAGATCCAAA Run19_sample1,1223,1969.0,Run19_sample1,70,088 BST Tac2 Gaba,70
1,AAACCCAGTACGGATG Run19_sample1,1827,3525.0,Run19_sample1,24,Unlabeled,24
2,AAACCCAGTTACGTAC Run19_sample1,846,1132.0,Run19_sample1,161,Unlabeled,161
3,AAACCCAGTTCAAACC Run19_sample1,2868,7582.0,Run19_sample1,184,141 PH-SUM Foxa1 Glut,184
4,AAACCCATCAGGGTAG Run19_sample1,1892,3861.0,Run19_sample1,268,097 PVHd-SBPV Six3 Prox1 Gaba,268
...,...,...,...,...,...,...,...
42216,TTTGTTGGTCATAGTC Run19_sample8,3095,6997.0,Run19_sample8,94,094 SCH Six6 Cdc14a Gaba,94
42217,TTTGTTGGTGAGAGGG Run19_sample8,1977,3914.0,Run19_sample8,25,037 DG Glut,25
42218,TTTGTTGGTGTTACTG Run19_sample8,1753,3628.0,Run19_sample8,304,214 IPN Otp Crisp1 Gaba,304
42219,TTTGTTGTCGGCCCAA Run19_sample8,1363,2131.0,Run19_sample8,58,Unlabeled,58


In [49]:
test = pd.read_csv(meta_folder + mn[0])['subclass_id_label_mapping']

In [50]:
barcodes = pd.read_csv(meta_folder + mn[0])['Unnamed: 0']

In [51]:
raw_lc = pd.read_csv(meta_folder + mn[0])['subclass_id_label_lc']

In [52]:
df_lc = pd.DataFrame(data = list(raw_lc), index = list(barcodes), columns = ['raw_lc'])

In [53]:
mode_df = pd.DataFrame(index = [a for a in range(len(test))], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    print(mn[j])
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_lc'])
    mode_df.loc[:,j] = dat

XT_metadata_subclass_250_cleaned_03122025_17.csv
XT_metadata_subclass_250_cleaned_03122025_2.csv
XT_metadata_subclass_250_cleaned_03122025_11.csv
XT_metadata_subclass_250_cleaned_03122025_22.csv
XT_metadata_subclass_250_cleaned_03122025_29.csv
XT_metadata_subclass_250_cleaned_03122025_21.csv
XT_metadata_subclass_250_cleaned_03122025_0.csv
XT_metadata_subclass_250_cleaned_03122025_16.csv
XT_metadata_subclass_250_cleaned_03122025_10.csv
XT_metadata_subclass_250_cleaned_03122025_15.csv
XT_metadata_subclass_250_cleaned_03122025_4.csv
XT_metadata_subclass_250_cleaned_03122025_3.csv
XT_metadata_subclass_250_cleaned_03122025_20.csv
XT_metadata_subclass_250_cleaned_03122025_5.csv
XT_metadata_subclass_250_cleaned_03122025_23.csv
XT_metadata_subclass_250_cleaned_03122025_8.csv
XT_metadata_subclass_250_cleaned_03122025_26.csv
XT_metadata_subclass_250_cleaned_03122025_18.csv
XT_metadata_subclass_250_cleaned_03122025_25.csv
XT_metadata_subclass_250_cleaned_03122025_27.csv
XT_metadata_subclass_250_c

In [54]:
fin = []
for item in mode_df.columns:
    fin.append(mode_df.loc[10000,item])

In [55]:
scipy.stats.mode(fin)

ModeResult(mode=array([131]), count=array([30]))

In [56]:
import re

mn = sorted(mn, key=lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)])

In [57]:
mn

['XT_metadata_subclass_250_cleaned_03122025_0.csv',
 'XT_metadata_subclass_250_cleaned_03122025_1.csv',
 'XT_metadata_subclass_250_cleaned_03122025_2.csv',
 'XT_metadata_subclass_250_cleaned_03122025_3.csv',
 'XT_metadata_subclass_250_cleaned_03122025_4.csv',
 'XT_metadata_subclass_250_cleaned_03122025_5.csv',
 'XT_metadata_subclass_250_cleaned_03122025_6.csv',
 'XT_metadata_subclass_250_cleaned_03122025_7.csv',
 'XT_metadata_subclass_250_cleaned_03122025_8.csv',
 'XT_metadata_subclass_250_cleaned_03122025_9.csv',
 'XT_metadata_subclass_250_cleaned_03122025_10.csv',
 'XT_metadata_subclass_250_cleaned_03122025_11.csv',
 'XT_metadata_subclass_250_cleaned_03122025_12.csv',
 'XT_metadata_subclass_250_cleaned_03122025_13.csv',
 'XT_metadata_subclass_250_cleaned_03122025_14.csv',
 'XT_metadata_subclass_250_cleaned_03122025_15.csv',
 'XT_metadata_subclass_250_cleaned_03122025_16.csv',
 'XT_metadata_subclass_250_cleaned_03122025_17.csv',
 'XT_metadata_subclass_250_cleaned_03122025_18.csv',
 'X

In [58]:
lsamap_mv = pd.DataFrame(index = [a for a in barcodes], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_mapping'])
    lsamap_mv.loc[:,j] = dat

In [20]:
#lsamap_m = lsamap_mv

In [87]:
#for quail
threshold = 0
a = 0
b = 0
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    a += 1
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '067 LSX Sall3 Pax6 Gaba':
            inp = ['cj_m066_m067']
        if inp[0] == '136 PMv-TMv Pitx2 Glut':
            inp = ['cj_m136_m138']
        if inp[0] == '099 SBPV-PVa Six6 Satb2 Gaba':
            inp = ['cj_m091_m099']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [113]:
#for anole
threshold = 0
a = 0
b = 0
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    a += 1
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '086 MPO-ADP Lhx8 Gaba':
            inp = ['ac_m058_m086']
        if inp[0] == '124 MPN-MPO-PVpo Hmx2 Glut':
            inp = ['ac_m124_m130']
        if inp[0] == '136 PMv-TMv Pitx2 Glut':
            inp = ['ac_m136_m138']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [59]:
#for xenopus
threshold = 0
a = 0
b = 0
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    a += 1
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '104 TU-ARH Otp Six6 Gaba':
            inp = ['xt_m098_m104']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    if lc == 182:
        print('made it')
        inp = ['104 TU-ARH Otp Six6 Gaba']
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

made it


In [26]:
#for zebrafish
threshold = 0
a = 0
b = 0
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    a += 1
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[1] != inp[2] and inp[1] != 'Unlabeled' and inp[2] != 'Unlabeled':
        print(inp[0],inp[1],inp[2],lc)
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [141]:
#for vole
threshold = .75
mapping_dict ={}
a = 0
b = 0
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    a += 1
    inp = ['Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    if  f_saturn_m[0]/len(ct_saturn_m)> threshold:
        inp[0] = mode_saturn_m[0]
    if f_samap_m[0]/len(ct_samap_m) > threshold:
        inp[1] = mode_samap_m[0]
    if inp[1] != inp[0] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
        if inp[0] == '077 CEA-BST Gal Avp Gaba':
            mapping_dict[lc] = 'mg_077_106'
        elif inp[0] == '105 TMd-DMH Foxd2 Gaba':
            mapping_dict[lc] = 'mg_105_107'
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [60]:
new_mapping = []
new_mapping_name = 'test'
for item in sam.adata.obs['eq_subclass_lc']:
    new_mapping.append(mapping_dict[item])
sam.adata.obs[new_mapping_name] = new_mapping

In [61]:
a = 0
fin_obs = []
for item in sam.adata.obs_names:
    if sam.adata.obs.loc[item,'test'] == sam.adata.obs.loc[item,'ss_subclass']:
        a += 1
        fin_obs.append(item)

In [62]:
a

42221

In [63]:
sam.adata

AnnData object with n_obs × n_vars = 42221 × 19621
    obs: 'n_genes', 'n_counts', 'key', 'leiden_clusters', 'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac', 'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN', 'eq_subclass_nounlabeled_nnm', 'eq_subclass_nounlabeled_nmm', 'neurotransmitter', 'ss_subclass_nounlabeled', 'ss_subclass_v2_nounlabeled', 'ss_subclass_v3_nounlabeled', 'ss_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled_nmm_micro', 'ss_subclass_nounlabeled_nmm_v2', 'ss_subclass_nounlabeled_nmm_v2_nn', 'ss_subclass_nounlabeled_nmm_v3_nn', 'neurotransmitter_v2', 'test', 'ss_subclass_old', 'ss_subclass_v4_nounlabeled', 'ss_subclass_v4_nounlabeled_nn', 'ss_subclass_nounlabeled_nmm_v4_nn', 'ss_subclass_nounlabeled_nmm_v4_nn_thresh30', 'ss_subclass_nounlabeled_nmm_cl_v4_nn', 'ss_subclass_old_v2'
    var: 'mask_genes', 'means', 'variances', 'weights', 'spatial_dispersions'
    uns: 'dimred_indices', 'path_to_file', 'preprocess_args', 'ranked_genes', 'run_arg

In [39]:
sam.save_anndata(fn)